In [1]:
pip install mediapipe opencv-python

In [2]:
import cv2
import numpy as np
import mediapipe as mp
# import argparse
import os
# from datetime import datetime
import time
import math
import csv

In [3]:
# requirements = [
#     "opencv-python",
#     "mediapipe",
#     "numpy"
# ]

# with open("requirements.txt", "w") as f:
#     for r in requirements:
#         f.write(r + "\n")

# print("requirements.txt created!")


In [4]:
mpDraw = mp.solutions.drawing_utils
mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence = 0.9)

# Drawing style helpers (optional customizations)
DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(255,0,0), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)

In [5]:
# ---------- Count Laps ----------
def check_leg(lms, left_flag, right_flag, lap_count):
    # left_lap_count
    if lms[25].y <= (lms[23].y + lms[24].y)/2:
        if not left_flag:
            lap_count += 1
            left_flag = True
    else:
        left_flag = False

    # right_lap_count
    if lms[26].y <= (lms[23].y + lms[24].y)/2:
        if not right_flag:
            lap_count += 1
            right_flag = True
    else:
        right_flag = False

    return left_flag, right_flag, lap_count
    

In [6]:
def side_lean(lms, frame_width, frame_height):
    dy = abs((lms[11].y - lms[12].y)*frame_height)
    dx = abs((lms[11].x - lms[12].x)*frame_width)
    tan_C = dy/dx
    return round(math.degrees(math.atan(tan_C)), 2)

In [7]:
def angle_vertical(lms, frame_width, frame_height, vertical_max):
    # dx = lms[23].x - lms[11].x
    # dy = lms[23].y - lms[11].y
    # dx = abs((lms[23].x+lms[24].x)/2 - (lms[11].x+lms[12].x)/2)
    dy = abs((lms[23].y+lms[24].y)/2 - (lms[11].y+lms[12].y)/2)
    r = dy
    # r = math.sqrt((dx*frame_width)**2 + (dy*frame_height)**2)
    if r > vertical_max:
        vertical_max = (r + 39*vertical_max)/40
        return 0, vertical_max
    cos_C = r / vertical_max
    cos_C = max(min(cos_C, 1), -1)   # clamp to avoid numerical errors
    angle = abs(math.degrees(math.acos(cos_C)))
    return round(angle, 2), vertical_max

In [8]:
def knee_height(lms, left_rising, left_peak, right_rising, right_peak, lks1, lks2, lks3, lks4, lks5, rks1, rks2, rks3, rks4, rks5):

    # LEFT leg
    L_HIP = lms[23].y
    L_KNEE = lms[25].y
    # L_ANKLE = lms[26].y

    # RIGHT leg
    R_HIP = lms[24].y
    R_KNEE = lms[26].y
    # R_ANKLE = lms[28].y

    # ---------- LEFT thresholds ----------
    L_T25 = (3*L_HIP + L_KNEE)*0.25
    L_T50 = (L_HIP + L_KNEE)*0.50
    L_T75 = (L_HIP + 3*L_KNEE)*0.25


    # ---------- RIGHT thresholds ----------
    R_T25 = (3*R_HIP + R_KNEE)*0.25
    R_T50 = (R_HIP + R_KNEE)*0.50
    R_T75 = (R_HIP + 3*R_KNEE)*0.25

    # ---------- LEFT category ----------
    if L_KNEE <= R_HIP:
        left_cat = 5
    elif L_KNEE <= R_T25:
        left_cat = 4
    elif L_KNEE <= R_T50:
        left_cat = 3
    elif L_KNEE <= R_T75:
        left_cat = 2
    else:
        left_cat = 1

    # ---------- RIGHT category ----------
    if R_KNEE <= L_HIP:
        right_cat = 5
    elif R_KNEE <= L_T25:
        right_cat = 4
    elif R_KNEE <= L_T50:
        right_cat = 3
    elif R_KNEE <= L_T75:
        right_cat = 2
    else:
        right_cat = 1

    isComplete = False
    # =======================================================
    #                   LEFT PEAK DETECTION
    # =======================================================

    # If knee begins to rise (enters above level 1)
    if left_cat > 1:
        left_rising = True
        left_peak = max(left_peak, left_cat)   # track highest level

    # If knee returns to ground (lks1) → finalize peak count
    if left_rising and left_cat == 1:
        if left_peak == 5: lks5 += 1
        elif left_peak == 4: lks4 += 1
        elif left_peak == 3: lks3 += 1
        elif left_peak == 2: lks2 += 1
        else: lks1 += 1

        # reset rise cycle
        left_rising = False
        left_peak = 1

        # step finished → add timestamp
        # step_times.append((time.time()-start_time))
        isComplete = True


    # =======================================================
    #                   RIGHT PEAK DETECTION
    # =======================================================

    if right_cat > 1:
        right_rising = True
        right_peak = max(right_peak, right_cat)

    if right_rising and right_cat == 1:
        if right_peak == 5: rks5 += 1
        elif right_peak == 4: rks4 += 1
        elif right_peak == 3: rks3 += 1
        elif right_peak == 2: rks2 += 1
        else: rks1 += 1

        right_rising = False
        right_peak = 1

        # step finished → add timestamp
        # step_times.append((time.time()-start_time))
        isComplete = True

    return left_rising, left_peak,right_rising, right_peak, lks1, lks2, lks3, lks4, lks5, rks1, rks2, rks3, rks4, rks5, isComplete



In [9]:
def arm_leg(lms, left_knee_flag, right_knee_flag, coordinate_count):
    # left_knee_count
    if lms[15].y <= (lms[12].y + lms[14].y)/2:
        if not left_knee_flag:
            coordinate_count += 1
            left_knee_flag = True
    else:
        left_knee_flag = False

    # right_lap_count
    if lms[16].y <= (lms[11].y + lms[13].y)/2:
        if not right_knee_flag:
            coordinate_count += 1
            right_knee_flag = True
    else:
        right_knee_flag = False

    return left_knee_flag, right_knee_flag, coordinate_count
    

In [10]:
def time_interwal(step_times):
    step_times_interwal = []
    length = len(step_times)
    for i in range(2,length):
        step_times_interwal.append(step_times[i]-step_times[i-1])

    return step_times_interwal
        

In [11]:
def pace(total_step_count, time_):
    return (total_step_count/time_)*60

In [12]:
def rhythm(step_times_interwal):
    return np.std(step_times_interwal)

In [13]:
def cal_high_knee_score(lap_count, avg_side_lean, avg_vertical_angle, left_knee_height_data, right_knee_height_data, coordinate_count, pace_, rhythm_):
    #code here
    # A. lap_count_score
    lap_count_score = 0
    if lap_count >= 300:
        lap_count_score = 5
    elif lap_count >= 240:
        lap_count_score = 4
    elif lap_count >= 180:
        lap_count_score = 3
    elif lap_count >= 120:
        lap_count_score = 2
    else:
        lap_count_score = 1

    # side_lean_score
    side_lean_score = 0
    if avg_side_lean < 2:
        side_lean_score = 4
    elif avg_side_lean < 4:
        side_lean_score = 3
    elif avg_side_lean < 6:
        side_lean_score = 2
    elif avg_side_lean < 8:
        side_lean_score = 1
    else:
        side_lean_score = 0

    # vertical_lean_score
    vertical_lean_score = 0
    if avg_vertical_angle < 8:
        vertical_lean_score = 4
    elif avg_vertical_angle < 15:
        vertical_lean_score = 3
    elif avg_vertical_angle < 25:
        vertical_lean_score = 2
    elif avg_vertical_angle < 30:
        vertical_lean_score = 1
    else:
        vertical_lean_score = 0
        
    # B. posture_score
    posture_score = side_lean_score + vertical_lean_score

    # C. knee_height
    knee_height_score = (5*(left_knee_height_data[4]+right_knee_height_data[4])+4*(left_knee_height_data[3]+right_knee_height_data[3]))
    knee_height_score += (3*(left_knee_height_data[2]+right_knee_height_data[2])+2*(left_knee_height_data[1]+right_knee_height_data[1]))
    knee_height_score += (left_knee_height_data[0]+right_knee_height_data[0])/300
    if knee_height_score > 0:
        knee_height_score = 5
        
    # D. arm-leg score
    arm_leg_score = 0
    if coordinate_count >= 250:
        arm_leg_score = 4
    elif coordinate_count >= 180:
        arm_leg_score = 3
    elif coordinate_count >= 120:
        arm_leg_score = 2
    elif coordinate_count >= 60:
        arm_leg_score = 1
    else:
        arm_leg_score = 0

    # pace_Score
    p_score = 0
    if pace_ >= 120:
        p_score = 1.5
    elif pace_ >= 100:
        p_score = 1
    elif pace_ >= 60:
        p_score = 0.5
    else:
        p_score = 0

    # rhythm_score
    r_score = 0
    if rhythm_ <= 0.05:
        r_score = 1.5
    elif rhythm_ <= 0.10:
        r_score = 1
    elif rhythm_ <= 0.15:
        r_score = 0.5
    else:
        r_score = 0

    # E. P_R_score
    pr_score = p_score + r_score

    _score = posture_score + knee_height_score + arm_leg_score + pr_score
    return _score, lap_count_score
    

In [14]:
def adjust_score_based_on_time(_score, video_time):
    # Full 3 minutes or more
    if video_time >= 180:
        return _score

    # 2–3 minutes → reduce score by 1
    elif 120 <= video_time < 180:
        return max(0, _score - 1)

    # 1–2 minutes → cap max at 2
    elif 60 <= video_time < 120:
        return min(_score, 2)

    # <1 minute → max = 1
    else:  # video_time < 60
        return min(_score, 1)


In [15]:
def predict_category(_score, lap_count_score):
    
    # Safety checks
    if _score < 0: _score = 0
    if _score > 20: _score = 20
    if lap_count_score < 0: lap_count_score = 0
    if lap_count_score > 5: lap_count_score = 5

    # -------------------------------
    # Top Tier Category (Excellent)
    # -------------------------------
    if _score >= 18:
        if lap_count_score >= 3:
            return "Excellent"
        else:
            return "Above Average"

    # -------------------------------
    # High-Mid Tier (Above Average / Average)
    # -------------------------------
    elif 14 <= _score <= 17:
        if lap_count_score >= 3:
            return "Above Average"
        elif lap_count_score == 2:
            return "Average"
        else:
            return "Below Average"

    # -------------------------------
    # Mid Tier (Average / Below Average)
    # -------------------------------
    elif 10 <= _score <= 13:
        if lap_count_score >= 3:
            return "Average"
        elif lap_count_score == 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lower Tier (Below Average)
    # -------------------------------
    elif 6 <= _score <= 9:
        if lap_count_score >= 2:
            return "Below Average"
        else:
            return "Poor"

    # -------------------------------
    # Lowest Tier (Poor)
    # -------------------------------
    else:  # _score 0–5
        return "Poor"

In [16]:
def add_data(row):
    # row must be a list: ["value1", "value2", ...]
    with open("high_knee_results.csv", "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(row)


In [17]:
# open("high_knee_results.csv", "w").close()
# data = ["ID", "Name", "Video_path", "Total_Lap", "MIN_side_lean", "MAX_side_lean", "Avg_side_lean(deg)", "MIN_vertical_angle", "MAX_vertical_angle", "Avg_vertical_angle(deg)", "Left_knee_height_data", "Right_knee_height_data", "Arm-Leg_coordination", "Pace", "Rhythm", "Score_20", "Category"]
# add_data(data)

In [18]:
def high_knee_jump(ID="DCXXXX", name="Life", path=0, data_print='Y'):
    
    # cap = cv2.VideoCapture(0)  # 0 = default camera
    cap = cv2.VideoCapture(path)
    
    # Define video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # remove in live case
    fps = cap.get(cv2.CAP_PROP_FPS)                            # frames per second
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))       # total frames
    video_time = frame_count / fps

    # print("FPS:", fps)
    # print("Total Frames:", frame_count)
    # print("Video Duration (seconds):", video_time_seconds)


    # save the output video
    # fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # out = cv2.VideoWriter(video_filename, fourcc, fps, (frame_width, frame_height))

    # lap count const
    lap_count = 0
    left_flag = False
    right_flag = False

    # posture const
    # side_lean
    side_lean_min = 100
    side_lean_max = 0
    side_lean_list = []

    # vertical_lean
    vertical_lean_min = 100
    vertical_lean_max = 0
    vertical_max = 0
    vertical_angle_list = []
    
    #  knee_height const
    # left_knee
    left_rising = False
    left_peak = 1
    lks1 = lks2 = lks3 = lks4 = lks5 = 0

    # left_knee
    right_rising = False
    right_peak = 1
    rks1 = rks2 = rks3 = rks4 = rks5 = 0 

    # arm_leg const
    coordinate_count = 0
    left_knee_flag = False
    right_knee_flag = False

    # pace const
    frame_idx = 0
    total_step_count = 0
    step_times = [0.000000000001]

    # rhythm const
    # step_times_interwal = []

    
    start_time = time.time()
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        # Convert BGR → RGB for MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Run pose detection
        results = pose.process(rgb)

        if results.pose_landmarks:
            # Enumerate all landmarks
            # for id, lm in enumerate(results.pose_landmarks.landmark):
            #     # Optional: Draw skeleton
            #     mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
                
            #     # Convert normalized landmark to pixel coordinates
            #     h, w, c = frame.shape
            #     cx, cy = int(lm.x * w), int(lm.y * h)
            #     # IDs to highlight: 23, 24, 25, 26
            #     if id in [23, 24, 25, 26]:
            #         # Draw circle on frame
            #         cv2.circle(frame, (cx, cy), 8, (0, 255, 0), -1)

            mpDraw.draw_landmarks(frame, results.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)
            lms = results.pose_landmarks.landmark

            # A. lap_count
            left_flag, right_flag, lap_count = check_leg(lms, left_flag, right_flag, lap_count)
            cv2.putText(frame, f"Accurate lapCount: {lap_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)

            # B. posture
            # B1. side_lean
            side_angle = side_lean(lms, frame_width, frame_height)
            cv2.putText(frame, f"side_lean: {side_angle}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            side_lean_list.append(side_angle)

            # B2.vertical_lean
            vertical_angle, vertical_max = angle_vertical(lms, frame_width, frame_height, vertical_max)
            cv2.putText(frame, f"verical_lean: {vertical_angle:.2f}", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            vertical_angle_list.append(vertical_angle)

            # C. knee_height
            left_rising, left_peak, right_rising, right_peak, lks1, lks2, lks3, lks4, lks5, rks1, rks2, rks3, rks4, rks5, isComplete = knee_height(lms, left_rising, left_peak, right_rising, right_peak, lks1, lks2, lks3, lks4, lks5, rks1, rks2, rks3, rks4, rks5)
            if isComplete:
                step_times.append(frame_idx/fps)
            cv2.putText(frame, f"left_knee_count: {lks1, lks2, lks3, lks4, lks5}", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"right_knee_count: {rks1, rks2, rks3, rks4, rks5}", (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)

            # D. arm-leg_coordination
            left_knee_flag, right_knee_flag, coordinate_count = arm_leg(lms, left_knee_flag, right_knee_flag, coordinate_count)
            cv2.putText(frame, f"arm-leg_coordination: {coordinate_count}", (50, 300), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            
        # out.write(frame)
        cv2.namedWindow("Resized Window", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("Resized Window", 900, 800)
        cv2.imshow("Resized Window", frame)
        # cv2.imshow("MyWindow", frame)


        # cv2.imshow("Press 'q' to stop early", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # print(f"Video_time: {video_time} and Calculated: {time.time()-start_time}")
    cap.release()
    # out.release()
    cv2.destroyAllWindows()
        
    # cal_time_interwal
    step_times_interwal = time_interwal(step_times)
    total_time = step_times[-1]

    # total step
    total_step_count = lks1 + lks2 + lks3 + lks4 + lks5 + rks1 + rks2 + rks3 + rks4 + rks5

    # E. pace and rhythm
    # E1. pace
    pace_ = int(pace(total_step_count, total_time))

    # E2. rhythm
    rhythm_ = round(rhythm(step_times_interwal), 4)

    # some parameter
    side_lean_min = round(min(side_lean_list), 2)
    side_lean_max = round(max(side_lean_list), 2)
    avg_side_lean = round(np.mean(side_lean_list), 2)
    
    vertical_lean_min = round(min(vertical_angle_list), 2)
    vertical_lean_max = round(max(vertical_angle_list), 2)
    avg_vertical_angle = round(np.mean(vertical_angle_list), 2)
    
    left_knee_height_data = [lks1, lks2, lks3, lks4, lks5]
    right_knee_height_data = [rks1, rks2, rks3, rks4, rks5]

    # show data
    if (data_print =='Y' or data_print == 'y'):
        print("printing data here------------->>")
        print(f"Accurate Lap Count: {lap_count}")
        print(f"Side Lean: ({side_lean_min}, {side_lean_max})")
        print(f"Vertical Lean: ({vertical_lean_min}, {vertical_lean_max}) and mean value is {avg_vertical_angle}")
        print(f"Left knee height: {left_knee_height_data}")
        print(f"Right knee height: {right_knee_height_data}")
        print(f"Arm-leg Coordination: {coordinate_count}")
        print(f"Pace: {pace_} lap/min")
        print(f"Rhythm: {rhythm_}")
        
    
    # score_20
    _score, lap_count_score = cal_high_knee_score(lap_count, avg_side_lean, avg_vertical_angle, left_knee_height_data, right_knee_height_data, coordinate_count, pace_, rhythm_)

    # adjusted_score(time)
    _score = round(adjust_score_based_on_time(_score, total_time), 2)
    
    # predict_category
    category = predict_category(_score, lap_count_score)

    # adding data
    data = [ID, name, path, lap_count, side_lean_min, side_lean_max, avg_side_lean, vertical_lean_min, vertical_lean_max, avg_vertical_angle, left_knee_height_data, right_knee_height_data, coordinate_count, pace_, rhythm_, _score, category]
    add_data(data)

    # print(step_times)
    # print("------------------------------------")
    # print(step_times_interwal)
    return _score, category
    

In [19]:

video_dir = "data2"  # change path

video_ext = (".mp4", ".avi", ".mov", ".mkv")

for video_name in os.listdir(video_dir):
    # if not video_name.lower().endswith(video_ext):
    #     continue

    path = os.path.join(video_dir, video_name)
    print(path)

    ID = "DC0001"
    name = "Nayan"
    data_print = "Y"

    high_knee_score, category = high_knee_jump(ID, name, path, data_print)
    print(f"Your High Knee Jump Score is--> {high_knee_score:.2f}/20 and Category--> {category}")

print("All videos read successfully.")


data2\Copy of Aarav jawadi 5th B.mp4


C:\Users\nayan\AppData\Local\Programs\Python\Python310\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


printing data here------------->>
Accurate Lap Count: 0
Side Lean: (0.0, 16.19)
Vertical Lean: (0, 60.47) and mean value is 26.74
Left knee height: [0, 7, 0, 0, 0]
Right knee height: [0, 8, 0, 0, 0]
Arm-leg Coordination: 10
Pace: 8 lap/min
Rhythm: 9.5889
Your High Knee Jump Score is--> 2.00/20 and Category--> Poor
data2\Copy of Abhed chaudhary 5th B.mp4
printing data here------------->>
Accurate Lap Count: 0
Side Lean: (0.0, 16.04)
Vertical Lean: (0, 63.1) and mean value is 31.32
Left knee height: [0, 16, 0, 0, 0]
Right knee height: [0, 18, 3, 2, 0]
Arm-leg Coordination: 35
Pace: 13 lap/min
Rhythm: 7.1342
Your High Knee Jump Score is--> 7.00/20 and Category--> Poor
data2\Copy of Ananya Bahuguna.mp4
printing data here------------->>
Accurate Lap Count: 4
Side Lean: (0.01, 84.43)
Vertical Lean: (0, 67.98) and mean value is 29.18
Left knee height: [0, 26, 0, 0, 1]
Right knee height: [0, 34, 6, 1, 3]
Arm-leg Coordination: 66
Pace: 27 lap/min
Rhythm: 3.2996
Your High Knee Jump Score is--> 8

In [20]:
# ID = "DC0001"
# name = "Nayan"
# path = "20251223_155629.mp4"
# data_print = "Y"
# high_knee_score, category = high_knee_jump(ID, name, path, data_print)
# print("-------------->>")
# print(f"Your High Knee Jump Score is--> {high_knee_score:.2f}/20 and Category--> {category}")

In [21]:
# ID = input("(DCXXXXX)Enter unique ID:")
# name = input("Name of Candidate:")
# path = input("Path of your Video:")
# # "data/img-5610-0q6jn2tf_fZX3eQZg.mov"
# # data/5124453_People_Person_3840x2160.mp4
# data_print = input("Wants to print data Y/N:")
# high_knee_score, category = high_knee_jump(ID, name, path, data_print)
# print("-------------->>")
# print(f"Your High Knee Jump Score is--> {high_knee_score:.2f}/20 and Category--> {category}")

In [22]:
# !jupyter nbconvert --to python high_knee.ipynb

In [24]:
# import pandas as pd
!pip install openpyxl


# Read CSV
df = pd.read_csv("high_knee_results.csv")

# Write to Excel
df.to_excel("high_knee_results.xlsx", index=False)

print("CSV converted to Excel successfully.")


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------